In [1]:
import os
os.chdir("/Users/ju/Projects/00_SMU/mqf_practice/QF603_Quant_Analysis_Fin_Markets")
os.getcwd()

'/Users/ju/Projects/00_SMU/mqf_practice/QF603_Quant_Analysis_Fin_Markets'

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRanker
import tensorflow as tf
from tensorflow import keras

# -------------------
# 1. Load Data
# -------------------

# Load daily_close_data.csv (wide format: Date, Ticker1, Ticker2, ...)
df_raw = pd.read_csv("./data/daily_close_data.csv")
print(f"Raw data shape: {df_raw.shape}")
# Melt to long format: columns = ['date', 'ticker', 'price']
df = df_raw.melt(id_vars=["Date"], var_name="ticker", value_name="price")
df = df.rename(columns={"Date": "date"})
# Sort for correct pct_change calculation
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)


Raw data shape: (756, 11)


In [3]:

# Calculate daily returns by ticker and add as a new column
df["ret"] = df.groupby("ticker")["price"].pct_change()

# Feature Engineering (momentum signals)

def momentum_features(data, lookbacks=[21, 63,126,252]):
    out = data.copy()
    for lb in lookbacks:
        out[f"mom_{lb}"] = out["price"].pct_change(lb)
        out[f"mom_{lb}_voladj"] = out["price"].pct_change(lb) / out["price"].pct_change().rolling(lb).std()
    return out

# Exclude grouping columns from apply to silence FutureWarning
df_features = df.groupby("ticker", group_keys=False)[["date", "price", "ret"]].apply(momentum_features)
df = df.join(df_features.drop(columns=['date', 'price', 'ret']))

# Label Construction (next month return)
df["label"] = df.groupby("ticker")["ret"].shift(-21)  # ~1 month ahead
print(f"Data shape after feature engineering: {df.shape}")

Data shape after feature engineering: (7560, 13)


In [4]:
# -------------------
# 4. Prepare Training Data
# -------------------
# After dropping NA, keep a copy of the continuous forward return
# preserve forward return, use period-month groups, create integer relevance
df_cleaned = df.dropna().copy()
df_cleaned["fwd_ret"] = df_cleaned["label"]  # preserve continuous forward return

df_cleaned["date"] = pd.to_datetime(df_cleaned["date"])
# keep a month column for downstream analysis (period M)
df_cleaned["month"] = df_cleaned["date"].dt.to_period("M")
# Create per-date rank labels from 1..10 across the cross-section of tickers for each date
def rank_to_10(s):
    # s is the forward return series for a single date (indexed by ticker rows)
    r = s.rank(method="first", ascending=True) - 1.0  # zero-based rank
    if r.max() > 0:
        # scale to 0..9, then +1 to obtain 1..10
        return (r / r.max() * 9).round().astype(int) + 1
    else:
        # if only one item on this date, assign rank 1
        return pd.Series(1, index=s.index)

df_cleaned["rank_label"] = df_cleaned.groupby("date")["fwd_ret"].transform(rank_to_10).astype(int)

# Use `rank_label` (1..10) for training (y). Use date as the query id so each date is a query/group.
X = df_cleaned[["mom_21", "mom_63","mom_126","mom_252"]].values
y = df_cleaned["rank_label"].values
qid = df_cleaned["date"].factorize()[0]  # group by date

print(f"(After raw 756 minus SMA 252 minus ret window 21)*10 X.shape: {X.shape}")
# unique dates as queries for train/test split
dates = np.sort(df_cleaned["date"].unique())
n_train = int(len(dates) * 0.8)

train_dates = dates[:n_train]
test_dates  = dates[n_train:]
# boolean masks by date
train_mask = df_cleaned["date"].isin(train_dates)
test_mask  = df_cleaned["date"].isin(test_dates)

# split
X_train = X[train_mask]
X_test  = X[test_mask]
y_train = y[train_mask]
y_test  = y[test_mask]
qid_train = qid[train_mask]
qid_test  = qid[test_mask]
print(f"Training data shape: {X_train.shape}, Test data shape: {X_test.shape}")

# --- Analysis of Training Data ---
# Get the part of the dataframe that corresponds to the training set (by date)
df_train = df_cleaned[df_cleaned.date.isin(train_dates)]
num_unique_stocks_train = df_train['ticker'].nunique()
print("\n--- Training Data Analysis ---")
print("------------------------------\n")
print(f"The {X_train.shape[0]} rows in X_train correspond to observations of these stocks across different dates.")
print(f"Number of unique stocks in the training data: {num_unique_stocks_train}")

(After raw 756 minus SMA 252 minus ret window 21)*10 X.shape: (4830, 4)
Training data shape: (3860, 4), Test data shape: (970, 4)

--- Training Data Analysis ---
------------------------------

The 3860 rows in X_train correspond to observations of these stocks across different dates.
Number of unique stocks in the training data: 10


In [5]:
# -------------------
# 5a. LambdaMART with XGBRanker
# -------------------
xgb_ranker = XGBRanker(
    objective="rank:pairwise",
    eval_metric="ndcg",
    eta=0.1,
    max_depth=6,
    tree_method="hist"
)

group_train = np.unique(qid_train, return_counts=True)[1]
group_test = np.unique(qid_test, return_counts=True)[1]

xgb_ranker.fit(X_train, y_train, group=group_train,
               eval_set=[(X_test, y_test)], eval_group=[group_test],
               verbose=True)

[0]	validation_0-ndcg:0.63810
[1]	validation_0-ndcg:0.61928
[2]	validation_0-ndcg:0.65677
[3]	validation_0-ndcg:0.66276
[4]	validation_0-ndcg:0.65418
[1]	validation_0-ndcg:0.61928
[2]	validation_0-ndcg:0.65677
[3]	validation_0-ndcg:0.66276
[4]	validation_0-ndcg:0.65418
[5]	validation_0-ndcg:0.65184
[6]	validation_0-ndcg:0.65217
[7]	validation_0-ndcg:0.65450
[8]	validation_0-ndcg:0.67467
[9]	validation_0-ndcg:0.66682
[5]	validation_0-ndcg:0.65184
[6]	validation_0-ndcg:0.65217
[7]	validation_0-ndcg:0.65450
[8]	validation_0-ndcg:0.67467
[9]	validation_0-ndcg:0.66682
[10]	validation_0-ndcg:0.67925
[11]	validation_0-ndcg:0.67707
[12]	validation_0-ndcg:0.66573
[13]	validation_0-ndcg:0.66612
[14]	validation_0-ndcg:0.66239
[10]	validation_0-ndcg:0.67925
[11]	validation_0-ndcg:0.67707
[12]	validation_0-ndcg:0.66573
[13]	validation_0-ndcg:0.66612
[14]	validation_0-ndcg:0.66239
[15]	validation_0-ndcg:0.65808
[16]	validation_0-ndcg:0.66452
[17]	validation_0-ndcg:0.66160
[18]	validation_0-ndcg:0.65

,objective,'rank:pairwise'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'ndcg'


In [15]:
X_train

array([[ 0.08124141,  0.13810134,  0.46283834,  0.78239877],
       [ 0.05142994,  0.14699374,  0.38950832,  0.7554053 ],
       [ 0.06564194,  0.12648967,  0.4110675 ,  0.76306118],
       ...,
       [-0.17424021, -0.01697875,  0.25307677,  0.45162403],
       [-0.15548484,  0.02217107,  0.26565203,  0.46440697],
       [-0.13260285, -0.01878929,  0.19022766,  0.46760803]],
      shape=(3860, 4))

In [24]:
qid_train[3200:]

array([112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124,
       125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137,
       138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150,
       151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163,
       164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176,
       177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189,
       190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202,
       203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215,
       216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228,
       229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241,
       242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254,
       255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267,
       268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280,
       281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 29

In [ ]:
# -------------------
# 6. Sharpe Ratio Comparison
# -------------------

# Get the part of the dataframe that corresponds to the test set
df_test = df_cleaned.iloc[-len(X_test):].copy()

# Get predictions from the models
df_test['xgb_score'] = xgb_ranker.predict(X_test)
# df_test['benchmark_score'] = X_test[:, 1]  # mom_126
# df_test['ranknet_score'] = ranknet.predict(X_test).flatten()

def calculate_sharpe_ratio(df, score_column):
    """Calculates the Sharpe ratio for a long-short quintile portfolio."""
    
    def get_portfolio_return(group):
        # Determine quintiles based on the score
        q = pd.qcut(group[score_column], 5, labels=False, duplicates='drop')
        
        # Get returns for top (long) and bottom (short) quintiles
        long_returns = group[q == 4]['label'].mean()
        short_returns = group[q == 0]['label'].mean()
        
        # Calculate long-short return for the period
        return long_returns - short_returns

    # Group by date and calculate the return for each period
    portfolio_returns = df.groupby('date').apply(get_portfolio_return)
    
    # Calculate annualized Sharpe ratio (assuming monthly returns)
    sharpe_ratio = (portfolio_returns.mean() / portfolio_returns.std()) * np.sqrt(12)
    return sharpe_ratio

# Calculate Sharpe ratio for each strategy
sharpe_xgb = calculate_sharpe_ratio(df_test, 'xgb_score')
# sharpe_benchmark = calculate_sharpe_ratio(df_test, 'benchmark_score')
# sharpe_ranknet = calculate_sharpe_ratio(df_test, 'ranknet_score')

print("\n--- Sharpe Ratio Comparison ---")
print(f"XGBRanker Sharpe Ratio:      {sharpe_xgb:.4f}")
# print(f"Benchmark Momentum Sharpe Ratio: {sharpe_benchmark:.4f}")
# print(f"RankNet Sharpe Ratio:        {sharpe_ranknet:.4f}")
print("-----------------------------")



--- Sharpe Ratio Comparison ---
XGBRanker Sharpe Ratio:      -0.0150
-----------------------------


/var/folders/h2/r7qn2m9n1zb6y_0q191gdqth0000gn/T/ipykernel_26228/3075955309.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  portfolio_returns = df.groupby('date').apply(get_portfolio_return)


In [8]:
df_test

,date,ticker,price,ret,mom_21,mom_21_voladj,mom_63,mom_63_voladj,mom_126,mom_126_voladj,mom_252,mom_252_voladj,label,fwd_ret,month,rank_label,xgb_score
6023,2022-11-25,MSFT,242.163330,-0.000364,0.072918,2.716008,-0.074238,-2.995072,-0.064632,-2.856439,-0.260723,-11.856629,-0.007414,-0.007414,2022-11,7,0.300242
6024,2022-11-28,MSFT,236.556656,-0.023152,0.069201,2.562459,-0.085921,-3.448470,-0.110833,-4.908834,-0.259812,-11.817844,-0.010255,-0.010255,2022-11,7,0.145505
6025,2022-11-29,MSFT,235.157440,-0.005915,0.021780,0.846906,-0.083518,-3.353429,-0.111638,-4.944161,-0.279381,-12.732343,0.027630,0.027630,2022-11,5,-0.035089
6026,2022-11-30,MSFT,249.648697,0.061624,0.102223,3.582758,-0.021459,-0.821611,-0.058799,-2.528819,-0.220997,-9.924072,-0.004938,-0.004938,2022-11,1,0.034208
6300,2020-12-31,TSLA,235.223328,0.015674,0.206769,5.393869,0.574594,15.748876,1.919225,38.171904,7.200506,127.769955,0.039271,0.039271,2020-12,9,-0.064542
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7534,2022-11-23,XOM,103.370689,-0.004992,0.081645,4.687148,0.155764,7.322107,0.200803,8.717613,0.871011,39.579086,0.026445,0.026445,2022-11,9,0.168195
7535,2022-11-25,XOM,103.006760,-0.003521,0.065161,3.747188,0.166051,7.831513,0.192366,8.349821,0.854144,38.807715,0.013894,0.013894,2022-11,10,0.116749
7536,2022-11-28,XOM,99.913162,-0.030033,0.029233,1.552142,0.105613,4.927339,0.145297,6.263223,0.863937,39.310855,-0.016426,-0.016426,2022-11,3,-0.040218
7537,2022-11-29,XOM,100.577393,0.006648,0.006593,0.370727,0.156992,7.538993,0.172006,7.430450,0.865971,39.402166,0.007566,0.007566,2022-11,3,-0.009554
